In [52]:
import pandas as pd

In [53]:
campaigns_df = pd.read_csv('data/campaigns_enriched.csv')


In [54]:
campaigns_df.columns

Index(['campaign_id', 'url', 'title', 'overview_text', 'details_text',
       'required_amount_raw', 'paid_amount_raw', 'left_amount_raw',
       'required_amount', 'paid_amount', 'left_amount', 'donations_count',
       'comments_count', 'updates_count', 'shares_count', 'image_url',
       'year_detected', 'year_source', 'date_status', 'scraped_at',
       'scrape_date', 'campaign_text', 'campaign_text_short',
       'left_amount_clean', 'amount_consistency_flag', 'is_overfunded',
       'overfunded_amount', 'year_confidence', 'has_uncertain_year',
       'year_label', 'funding_ratio', 'funding_status', 'engagement_total',
       'text_length', 'has_details_text', 'has_default_image',
       'title_normalized', 'campaign_series_key', 'campaign_theme',
       'beneficiary_group', 'location_mentions', 'record_quality_score',
       'search_chunks'],
      dtype='str')

In [55]:
campaigns_df.head()

,campaign_id,url,title,overview_text,details_text,required_amount_raw,paid_amount_raw,left_amount_raw,required_amount,paid_amount,...,text_length,has_details_text,has_default_image,title_normalized,campaign_series_key,campaign_theme,beneficiary_group,location_mentions,record_quality_score,search_chunks
0,1,https://molhamteam.com/en/campaigns/1,Razan's Fund for Cancer Patients,Razan's Fund for Cancer Patients International...,International relief organizations have discon...,66176.00,66177.00,-1.00,66176.00,66177.00,...,747,True,True,razan s fund for cancer patients,razan s fund for cancer patients,medical,patients,"[""Jordan"", ""Lebanon"", ""Syria"", ""Turkey""]",97,"[{""chunk_id"": ""1-0"", ""campaign_id"": 1, ""title""..."
1,2,https://molhamteam.com/en/campaigns/2,Education Fund,Hundreds of thousands of Syrian children no lo...,NaN,99777.00,99777.00,0.00,99777.00,99777.00,...,328,False,True,education fund,education fund,education,students,"[""Syria""]",87,"[{""chunk_id"": ""2-0"", ""campaign_id"": 2, ""title""..."
2,3,https://molhamteam.com/en/campaigns/3,Medical Emergencies Fund,The Team receives a number of medical cases th...,NaN,49551.00,49551.00,0.00,49551.00,49551.00,...,198,False,True,medical emergencies fund,medical emergencies fund,medical,patients,[],87,"[{""chunk_id"": ""3-0"", ""campaign_id"": 3, ""title""..."
3,4,https://molhamteam.com/en/campaigns/4,Dialysis Fund,This fund was established to help impoverished...,NaN,11591.00,11591.00,0.00,11591.00,11591.00,...,186,False,False,dialysis fund,dialysis fund,medical,patients,[],90,"[{""chunk_id"": ""4-0"", ""campaign_id"": 4, ""title""..."
4,5,https://molhamteam.com/en/campaigns/5,Drinking Water Campaign,A large number of families in Aleppo lost thei...,NaN,41343.00,41343.00,0.00,41343.00,41343.00,...,260,False,True,drinking water campaign,drinking water campaign,water,families,"[""Aleppo""]",87,"[{""chunk_id"": ""5-0"", ""campaign_id"": 5, ""title""..."


In [56]:
print(campaigns_df.shape)
unique_ids = campaigns_df['campaign_id'].nunique()
print("Number of unique IDs DataFrame:", unique_ids)

null_count = campaigns_df['campaign_id'].isnull().sum()
print("Number of null values in campaign_id:", null_count)

duplicate_rows = campaigns_df[campaigns_df.duplicated(subset=['campaign_id','year_detected'])]
print("Number of duplicate rows with same campaign and year detected :", len(duplicate_rows))

(823, 43)
Number of unique IDs DataFrame: 823
Number of null values in campaign_id: 0
Number of duplicate rows with same campaign and year detected : 0


# Data Cleaning & Integration

### [1]

In [59]:
campaigns_columns_to_keep = ['campaign_id', 'url', 'title', 'overview_text', 'details_text',
       'required_amount', 'paid_amount', 'left_amount', 'donations_count',
       'comments_count', 'updates_count', 'shares_count', 'image_url',
       'year_detected', 'year_source', 'date_status', 'scraped_at',
       'scrape_date', 'campaign_text', 'campaign_text_short',
       'left_amount_clean', 'amount_consistency_flag', 'is_overfunded',
       'overfunded_amount', 'year_confidence', 'has_uncertain_year',
       'year_label', 'funding_ratio', 'funding_status', 'engagement_total',
       'text_length', 'has_details_text', 'has_default_image',
       'title_normalized', 'campaign_series_key', 'campaign_theme',
       'beneficiary_group', 'location_mentions', 'record_quality_score',
       'search_chunks']
campaigns_df_filtered = campaigns_df[campaigns_columns_to_keep]
campaigns_df_filtered.head()

,campaign_id,url,title,overview_text,details_text,required_amount,paid_amount,left_amount,donations_count,comments_count,...,text_length,has_details_text,has_default_image,title_normalized,campaign_series_key,campaign_theme,beneficiary_group,location_mentions,record_quality_score,search_chunks
0,1,https://molhamteam.com/en/campaigns/1,Razan's Fund for Cancer Patients,Razan's Fund for Cancer Patients International...,International relief organizations have discon...,66176.00,66177.00,-1.00,509,0,...,747,True,True,razan s fund for cancer patients,razan s fund for cancer patients,medical,patients,"[""Jordan"", ""Lebanon"", ""Syria"", ""Turkey""]",97,"[{""chunk_id"": ""1-0"", ""campaign_id"": 1, ""title""..."
1,2,https://molhamteam.com/en/campaigns/2,Education Fund,Hundreds of thousands of Syrian children no lo...,NaN,99777.00,99777.00,0.00,239,0,...,328,False,True,education fund,education fund,education,students,"[""Syria""]",87,"[{""chunk_id"": ""2-0"", ""campaign_id"": 2, ""title""..."
2,3,https://molhamteam.com/en/campaigns/3,Medical Emergencies Fund,The Team receives a number of medical cases th...,NaN,49551.00,49551.00,0.00,476,0,...,198,False,True,medical emergencies fund,medical emergencies fund,medical,patients,[],87,"[{""chunk_id"": ""3-0"", ""campaign_id"": 3, ""title""..."
3,4,https://molhamteam.com/en/campaigns/4,Dialysis Fund,This fund was established to help impoverished...,NaN,11591.00,11591.00,0.00,149,0,...,186,False,False,dialysis fund,dialysis fund,medical,patients,[],90,"[{""chunk_id"": ""4-0"", ""campaign_id"": 4, ""title""..."
4,5,https://molhamteam.com/en/campaigns/5,Drinking Water Campaign,A large number of families in Aleppo lost thei...,NaN,41343.00,41343.00,0.00,124,0,...,260,False,True,drinking water campaign,drinking water campaign,water,families,"[""Aleppo""]",87,"[{""chunk_id"": ""5-0"", ""campaign_id"": 5, ""title""..."


### [2]

In [60]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

campaigns_df_filtered.describe()

,campaign_id,required_amount,paid_amount,left_amount,donations_count,comments_count,updates_count,shares_count,year_detected,left_amount_clean,overfunded_amount,funding_ratio,engagement_total,text_length,record_quality_score
count,823.00,821.00,818.00,815.00,823.00,823.00,823.00,823.00,808.00,816.00,816.00,816.00,823.00,823.00,823.00
mean,440.33,186016.65,111036.34,75940.72,1228.86,0.54,4.07,3.22,2022.04,75847.76,3492.94,0.87,1236.69,900.54,92.73
std,259.79,967901.83,535446.04,660303.44,7987.55,2.28,8.82,8.88,2.88,659904.99,30007.66,0.61,7990.02,625.55,6.52
min,1.00,5.00,0.00,-709675.00,0.00,0.00,0.00,0.00,2015.00,-709675.00,0.00,0.00,0.00,96.00,35.00
25%,219.50,20475.00,11809.00,0.00,130.50,0.00,0.00,0.00,2020.00,0.00,0.00,0.71,133.00,447.00,90.00
50%,435.00,51560.00,41202.50,0.00,414.00,0.00,0.00,0.00,2023.00,0.00,0.00,1.00,422.00,696.00,90.00
75%,657.50,118562.00,104495.50,14345.50,1123.00,0.00,4.00,3.00,2024.00,14233.25,0.00,1.00,1126.50,1214.00,100.00
max,910.00,20000000.00,13878058.00,14657200.00,222431.00,29.00,75.00,140.00,2026.00,14657250.00,709675.00,10.17,222443.00,5997.00,100.00


### [3]

In [61]:
def print_missing_values(df):
    info_df = pd.DataFrame()
    info_df['missing_val'] = df.isnull().sum()
    info_df['unknown_count'] = (df == 'UNKNOWN').sum()

    info_df['missing_val_ratio'] = (info_df['missing_val'] / df.shape[0] * 100)
    info_df['unknown_ratio'] = (info_df['unknown_count'] / df.shape[0] * 100)
    info_df['both_values'] = info_df['missing_val_ratio'] + info_df['unknown_ratio']

    info_df = info_df.sort_values(by='both_values', ascending=False)
    info_df = info_df[info_df['both_values'] > 0]



    return info_df.style.format(thousands=',', precision=2)

In [62]:
print_missing_values(campaigns_df_filtered)

,missing_val,unknown_count,missing_val_ratio,unknown_ratio,both_values
details_text,500,0,60.75,0.00,60.75
year_detected,15,0,1.82,0.00,1.82
left_amount,8,0,0.97,0.00,0.97
overfunded_amount,7,0,0.85,0.00,0.85
funding_ratio,7,0,0.85,0.00,0.85
left_amount_clean,7,0,0.85,0.00,0.85
paid_amount,5,0,0.61,0.00,0.61
required_amount,2,0,0.24,0.00,0.24
year_source,1,0,0.12,0.00,0.12
